## Program 1: Linux Process System Calls

Create a child process with fork(), replace its program with exec(), and wait for it to finish.


In [ ]:
import os
print('Parent started, PID =', os.getpid(), flush=True)
child_pid = os.fork()
if child_pid == 0:
    print('\nChild: PID =', os.getpid(), ', Parent PID =', os.getppid())
    print("Child: now running 'date' using exec()")
    os.execvp('date', ['date'])
    print('This line never runs')
else:
    print('Parent: created child with PID =', child_pid)
    child_pid, child_status = os.wait()
    print('\nParent: child', child_pid, 'finished')
    print('Parent: exit status =', os.WEXITSTATUS(child_status))
    print('Parent: done')

## Program 2: Thread Creation with Python threading

This example uses Python threads. It is not a direct POSIX pthreads implementation.


In [ ]:
import threading
import time
thread_results = {}

def run_worker(worker_id):
    print(f'Thread {worker_id}: started (name = {threading.current_thread().name})')
    time.sleep(1)
    print(f'Thread {worker_id}: finished')
    thread_results[worker_id] = worker_id * 10
thread_count = 3
worker_threads = []
print(f'Main: creating {thread_count} threads')
for i in range(1, thread_count + 1):
    worker_thread = threading.Thread(target=run_worker, args=(i,), name=f'Worker-{i}')
    worker_threads.append(worker_thread)
    worker_thread.start()
for worker_thread in worker_threads:
    worker_thread.join()
for tid, value in thread_results.items():
    print(f'Main: thread {tid} returned {value}')
print('Main: all threads completed')

## Program 3: CPU Scheduling: FCFS, SJF, Priority and Round Robin

Compare waiting time and turnaround time for four CPU scheduling strategies.

**Sample input for the saved run:**

```text
4
1
0
5
2
2
1
3
1
3
2
8
3
4
4
3
4
1
```

For SJF and Priority, change the final choice to `2` or `3`. For Round Robin, use choice `4` and add time quantum `2` on the next line.


In [ ]:
PROCESS_LIMIT = 20

def printResult(processes):
    n = len(processes)
    total_waiting_time = 0
    total_turnaround_time = 0
    print('\nPID\tAT\tBT\tCT\tTAT\tWT')
    for job in processes:
        print(f"{job['pid']}\t{job['at']}\t{job['bt']}\t{job['ct']}\t{job['tat']}\t{job['wt']}")
        total_waiting_time += job['wt']
        total_turnaround_time += job['tat']
    print(f'Average Waiting Time    = {total_waiting_time / n:.2f}')
    print(f'Average Turnaround Time = {total_turnaround_time / n:.2f}')

def fcfs(processes):
    current_time = 0
    for job in processes:
        if current_time < job['at']:
            current_time = job['at']
        current_time += job['bt']
        job['ct'] = current_time
        job['tat'] = job['ct'] - job['at']
        job['wt'] = job['tat'] - job['bt']
    printResult(processes)

def sjf(processes):
    n = len(processes)
    completed_count = 0
    current_time = 0
    completed_flags = [False] * n
    while completed_count < n:
        selected_index = -1
        shortest_burst = float('inf')
        for i in range(n):
            if not completed_flags[i] and processes[i]['at'] <= current_time and (processes[i]['bt'] < shortest_burst):
                shortest_burst = processes[i]['bt']
                selected_index = i
        if selected_index == -1:
            current_time += 1
            continue
        current_time += processes[selected_index]['bt']
        processes[selected_index]['ct'] = current_time
        processes[selected_index]['tat'] = processes[selected_index]['ct'] - processes[selected_index]['at']
        processes[selected_index]['wt'] = processes[selected_index]['tat'] - processes[selected_index]['bt']
        completed_flags[selected_index] = True
        completed_count += 1
    printResult(processes)

def priorityScheduling(processes):
    n = len(processes)
    completed_count = 0
    current_time = 0
    completed_flags = [False] * n
    while completed_count < n:
        selected_index = -1
        best_priority = float('inf')
        for i in range(n):
            if not completed_flags[i] and processes[i]['at'] <= current_time and (processes[i]['pr'] < best_priority):
                best_priority = processes[i]['pr']
                selected_index = i
        if selected_index == -1:
            current_time += 1
            continue
        current_time += processes[selected_index]['bt']
        processes[selected_index]['ct'] = current_time
        processes[selected_index]['tat'] = processes[selected_index]['ct'] - processes[selected_index]['at']
        processes[selected_index]['wt'] = processes[selected_index]['tat'] - processes[selected_index]['bt']
        completed_flags[selected_index] = True
        completed_count += 1
    printResult(processes)

def roundRobin(processes, time_quantum):
    n = len(processes)
    current_time = 0
    completed_count = 0
    for job in processes:
        job['rem'] = job['bt']
    queue = []
    queued_flags = [False] * n
    for i in range(n):
        if processes[i]['at'] <= current_time:
            queue.append(i)
            queued_flags[i] = True
    queue_front = 0
    while completed_count < n:
        if queue_front == len(queue):
            current_time += 1
            for i in range(n):
                if not queued_flags[i] and processes[i]['rem'] > 0 and (processes[i]['at'] <= current_time):
                    queue.append(i)
                    queued_flags[i] = True
            continue
        selected_index = queue[queue_front]
        queue_front += 1
        time_slice = min(processes[selected_index]['rem'], time_quantum)
        current_time += time_slice
        processes[selected_index]['rem'] -= time_slice
        for i in range(n):
            if not queued_flags[i] and processes[i]['rem'] > 0 and (processes[i]['at'] <= current_time) and (i != selected_index):
                queue.append(i)
                queued_flags[i] = True
        if processes[selected_index]['rem'] > 0:
            queue.append(selected_index)
        else:
            processes[selected_index]['ct'] = current_time
            processes[selected_index]['tat'] = processes[selected_index]['ct'] - processes[selected_index]['at']
            processes[selected_index]['wt'] = processes[selected_index]['tat'] - processes[selected_index]['bt']
            completed_count += 1
            queued_flags[selected_index] = False
    printResult(processes)
if __name__ == '__main__':
    n = int(input('Enter number of processes: '))
    processes = []
    for i in range(n):
        print(f'\nEnter details for Process {i + 1}')
        process_id = int(input('Enter Process ID: '))
        arrival_time = int(input('Enter Arrival Time: '))
        burst_time = int(input('Enter Burst Time: '))
        priority = int(input('Enter Priority: '))
        processes.append({'pid': process_id, 'at': arrival_time, 'bt': burst_time, 'pr': priority, 'ct': 0, 'tat': 0, 'wt': 0, 'rem': 0})
    print('\nChoose Scheduling Algorithm:')
    print('1. FCFS')
    print('2. SJF')
    print('3. Priority Scheduling')
    print('4. Round Robin')
    choice = int(input('Enter your choice: '))
    if choice == 1:
        fcfs([job.copy() for job in processes])
    elif choice == 2:
        sjf([job.copy() for job in processes])
    elif choice == 3:
        priorityScheduling([job.copy() for job in processes])
    elif choice == 4:
        time_quantum = int(input('Enter Time Quantum: '))
        if time_quantum <= 0:
            print('Time Quantum must be greater than zero.')
        else:
            roundRobin([job.copy() for job in processes], time_quantum)
    else:
        print('Invalid choice!')

## Program 4: Producer-Consumer Synchronization

A bounded circular buffer is shared safely using semaphores.


In [ ]:
import threading
import time
BUFFER_CAPACITY = 5
ITEM_COUNT = 10
shared_buffer = [0] * BUFFER_CAPACITY
write_position = 0
read_position = 0
empty_slots = threading.Semaphore(BUFFER_CAPACITY)
filled_slots = threading.Semaphore(0)
buffer_mutex = threading.Semaphore(1)

def produce_items():
    global write_position
    for i in range(1, ITEM_COUNT + 1):
        empty_slots.acquire()
        buffer_mutex.acquire()
        shared_buffer[write_position] = i
        print(f'Producer produced item {i} at slot {write_position}')
        write_position = (write_position + 1) % BUFFER_CAPACITY
        buffer_mutex.release()
        filled_slots.release()
        time.sleep(0.1)

def consume_items():
    global read_position
    for i in range(1, ITEM_COUNT + 1):
        filled_slots.acquire()
        buffer_mutex.acquire()
        item = shared_buffer[read_position]
        print(f'\t\tConsumer consumed item {item} from slot {read_position}')
        read_position = (read_position + 1) % BUFFER_CAPACITY
        buffer_mutex.release()
        empty_slots.release()
        time.sleep(0.15)

def main():
    producer_thread = threading.Thread(target=produce_items)
    consumer_thread = threading.Thread(target=consume_items)
    producer_thread.start()
    consumer_thread.start()
    producer_thread.join()
    consumer_thread.join()
    print('All items produced and consumed.')
if __name__ == '__main__':
    main()

## Program 5: Readers-Writers Synchronization

Multiple readers may access the shared value together, while writers require exclusive access.


In [ ]:
import threading
import time
reader_count_mutex = threading.Semaphore(1)
resource_semaphore = threading.Semaphore(1)
active_readers = 0
shared_value = 0

def read_shared_value(worker_id):
    global active_readers
    reader_count_mutex.acquire()
    active_readers += 1
    if active_readers == 1:
        resource_semaphore.acquire()
    reader_count_mutex.release()
    print(f'Reader {worker_id}: read shared_data = {shared_value}')
    time.sleep(0.2)
    reader_count_mutex.acquire()
    active_readers -= 1
    if active_readers == 0:
        resource_semaphore.release()
    reader_count_mutex.release()

def write_shared_value(worker_id):
    global shared_value
    resource_semaphore.acquire()
    shared_value += 1
    print(f'Writer {worker_id}: wrote shared_data = {shared_value}')
    time.sleep(0.3)
    resource_semaphore.release()

def main():
    reader_threads = []
    writer_threads = []
    first_writer = threading.Thread(target=write_shared_value, args=(1,))
    writer_threads.append(first_writer)
    first_writer.start()
    for i in range(1, 6):
        reader_thread = threading.Thread(target=read_shared_value, args=(i,))
        reader_threads.append(reader_thread)
        reader_thread.start()
    second_writer = threading.Thread(target=write_shared_value, args=(2,))
    writer_threads.append(second_writer)
    second_writer.start()
    for reader_thread in reader_threads:
        reader_thread.join()
    for writer_thread in writer_threads:
        writer_thread.join()
    print('All readers and writers have finished.')
if __name__ == '__main__':
    main()

## Program 6: Dining Philosophers

A room semaphore limits the number of competing philosophers and prevents circular wait.


In [ ]:
import threading
import time
PHILOSOPHER_COUNT = 5
fork_semaphores = [threading.Semaphore(1) for _ in range(PHILOSOPHER_COUNT)]
entry_semaphore = threading.Semaphore(PHILOSOPHER_COUNT - 1)

def dine(philosopher_id):
    left_fork = philosopher_id
    right_fork = (philosopher_id + 1) % PHILOSOPHER_COUNT
    for meal_round in range(2):
        print(f'Philosopher {philosopher_id} is thinking')
        time.sleep(0.1)
        entry_semaphore.acquire()
        fork_semaphores[left_fork].acquire()
        fork_semaphores[right_fork].acquire()
        print(f'Philosopher {philosopher_id} is eating (fork {left_fork} and fork {right_fork})')
        time.sleep(0.15)
        fork_semaphores[right_fork].release()
        fork_semaphores[left_fork].release()
        entry_semaphore.release()
        print(f'Philosopher {philosopher_id} finished eating and put down forks')

def main():
    philosopher_threads = []
    for i in range(PHILOSOPHER_COUNT):
        philosopher_thread = threading.Thread(target=dine, args=(i,))
        philosopher_threads.append(philosopher_thread)
        philosopher_thread.start()
    for philosopher_thread in philosopher_threads:
        philosopher_thread.join()
    print('All philosophers have finished eating.')
if __name__ == '__main__':
    main()

## Program 7: Banker's Algorithm

Calculate the Need matrix, check for a safe sequence, and evaluate a resource request.

**Sample input for the saved run:**

```text
3 3 2
7 5 3
3 2 2
9 0 2
2 2 2
4 3 3
0 1 0
2 0 0
3 0 2
2 1 1
0 2 2
1
1 0 2
```


In [ ]:
PROCESS_COUNT = 5
RESOURCE_COUNT = 3
available_resources = [0] * RESOURCE_COUNT
maximum_demand = [[0] * RESOURCE_COUNT for _ in range(PROCESS_COUNT)]
current_allocation = [[0] * RESOURCE_COUNT for _ in range(PROCESS_COUNT)]
remaining_need = [[0] * RESOURCE_COUNT for _ in range(PROCESS_COUNT)]

def calculate_need():
    for i in range(PROCESS_COUNT):
        for j in range(RESOURCE_COUNT):
            remaining_need[i][j] = maximum_demand[i][j] - current_allocation[i][j]

def check_safe_state():
    work_resources = available_resources.copy()
    finished_flags = [False] * PROCESS_COUNT
    safe_sequence = []
    completed_count = 0
    while completed_count < PROCESS_COUNT:
        progress_made = False
        for i in range(PROCESS_COUNT):
            if not finished_flags[i]:
                j = 0
                while j < RESOURCE_COUNT and remaining_need[i][j] <= work_resources[j]:
                    j += 1
                if j == RESOURCE_COUNT:
                    for k in range(RESOURCE_COUNT):
                        work_resources[k] += current_allocation[i][k]
                    safe_sequence.append(i)
                    finished_flags[i] = True
                    completed_count += 1
                    progress_made = True
        if not progress_made:
            return (False, [])
    return (True, safe_sequence)

def request_resources(process_id, request_vector):
    if process_id < 0 or process_id >= PROCESS_COUNT:
        print('Error: Invalid process number.')
        return False
    for j in range(RESOURCE_COUNT):
        if request_vector[j] > remaining_need[process_id][j]:
            print('Error: Process has exceeded its maximum claim.')
            return False
        if request_vector[j] > available_resources[j]:
            print('Request denied: Not enough resources available.')
            return False
    for j in range(RESOURCE_COUNT):
        available_resources[j] -= request_vector[j]
        current_allocation[process_id][j] += request_vector[j]
        remaining_need[process_id][j] -= request_vector[j]
    safe_state, safe_sequence = check_safe_state()
    if safe_state:
        print('Request can be granted safely.')
        print('Safe sequence:', ' '.join((f'P{i}' for i in safe_sequence)))
        return True
    else:
        for j in range(RESOURCE_COUNT):
            available_resources[j] += request_vector[j]
            current_allocation[process_id][j] -= request_vector[j]
            remaining_need[process_id][j] += request_vector[j]
        print('Request denied: Would lead to an unsafe state.')
        return False

def main():
    global available_resources, maximum_demand, current_allocation
    print(f'Enter available resources ({RESOURCE_COUNT} values):')
    available_resources = list(map(int, input().split()))
    if len(available_resources) != RESOURCE_COUNT or any((x < 0 for x in available_resources)):
        print('Invalid available resources.')
        return
    print(f'Enter Maximum matrix ({PROCESS_COUNT} processes x {RESOURCE_COUNT} resources):')
    for i in range(PROCESS_COUNT):
        maximum_demand[i] = list(map(int, input(f'P{i}: ').split()))
        if len(maximum_demand[i]) != RESOURCE_COUNT or any((x < 0 for x in maximum_demand[i])):
            print('Invalid Maximum matrix.')
            return
    print('Enter Allocation matrix:')
    for i in range(PROCESS_COUNT):
        current_allocation[i] = list(map(int, input(f'P{i}: ').split()))
        if len(current_allocation[i]) != RESOURCE_COUNT or any((x < 0 for x in current_allocation[i])):
            print('Invalid Allocation matrix.')
            return
    for i in range(PROCESS_COUNT):
        for j in range(RESOURCE_COUNT):
            if current_allocation[i][j] > maximum_demand[i][j]:
                print('Error: Allocation exceeds maximum claim.')
                return
    calculate_need()
    print('\nNeed Matrix:')
    for i in range(PROCESS_COUNT):
        print(f'P{i}: {remaining_need[i]}')
    safe_state, safe_sequence = check_safe_state()
    if safe_state:
        print('\nSystem is in a SAFE state.')
        print('Safe sequence:', ' '.join((f'P{i}' for i in safe_sequence)))
    else:
        print('\nSystem is in an UNSAFE state.')
    try:
        process_id = int(input('\nEnter process number requesting resources (0-4): '))
        request_vector = list(map(int, input(f'Enter request vector ({RESOURCE_COUNT} values): ').split()))
        if len(request_vector) != RESOURCE_COUNT or any((x < 0 for x in request_vector)):
            print('Invalid request vector.')
            return
        if not safe_state:
            print('Request denied: System is already in an unsafe state.')
            return
        request_resources(process_id, request_vector)
    except ValueError:
        print('Invalid input. Please enter integers.')
if __name__ == '__main__':
    main()

## Program 8: Contiguous Memory Allocation

Compare First Fit, Best Fit and Worst Fit using the same block and process sizes.

**Sample input for the saved run:**

```text
5
110 520 200 340 620
4
212 412 112 426
```


In [ ]:
ITEM_LIMIT = 20

def first_fit(block_sizes, process_sizes):
    block_count = len(block_sizes)
    process_count = len(process_sizes)
    remaining_blocks = block_sizes.copy()
    assignments = [-1] * process_count
    for i in range(process_count):
        for j in range(block_count):
            if remaining_blocks[j] >= process_sizes[i]:
                assignments[i] = j
                remaining_blocks[j] -= process_sizes[i]
                break
    print('\n--- First Fit ---')
    print('Process No.\tProcess Size\tBlock No.')
    for i in range(process_count):
        print(f'P{i + 1}\t\t{process_sizes[i]}\t\t', end='')
        if assignments[i] != -1:
            print(f'B{assignments[i] + 1}')
        else:
            print('Not Allocated')

def best_fit(block_sizes, process_sizes):
    block_count = len(block_sizes)
    process_count = len(process_sizes)
    remaining_blocks = block_sizes.copy()
    assignments = [-1] * process_count
    for i in range(process_count):
        best_index = -1
        for j in range(block_count):
            if remaining_blocks[j] >= process_sizes[i]:
                if best_index == -1 or remaining_blocks[j] < remaining_blocks[best_index]:
                    best_index = j
        if best_index != -1:
            assignments[i] = best_index
            remaining_blocks[best_index] -= process_sizes[i]
    print('\n--- Best Fit ---')
    print('Process No.\tProcess Size\tBlock No.')
    for i in range(process_count):
        print(f'P{i + 1}\t\t{process_sizes[i]}\t\t', end='')
        if assignments[i] != -1:
            print(f'B{assignments[i] + 1}')
        else:
            print('Not Allocated')

def worst_fit(block_sizes, process_sizes):
    block_count = len(block_sizes)
    process_count = len(process_sizes)
    remaining_blocks = block_sizes.copy()
    assignments = [-1] * process_count
    for i in range(process_count):
        largest_index = -1
        for j in range(block_count):
            if remaining_blocks[j] >= process_sizes[i]:
                if largest_index == -1 or remaining_blocks[j] > remaining_blocks[largest_index]:
                    largest_index = j
        if largest_index != -1:
            assignments[i] = largest_index
            remaining_blocks[largest_index] -= process_sizes[i]
    print('\n--- Worst Fit ---')
    print('Process No.\tProcess Size\tBlock No.')
    for i in range(process_count):
        print(f'P{i + 1}\t\t{process_sizes[i]}\t\t', end='')
        if assignments[i] != -1:
            print(f'B{assignments[i] + 1}')
        else:
            print('Not Allocated')
if __name__ == '__main__':
    block_count = int(input('Enter number of memory blocks: '))
    if block_count < 1 or block_count > ITEM_LIMIT:
        print('Number of blocks must be between 1 and 20.')
        raise SystemExit
    block_sizes = list(map(int, input('Enter sizes of blocks: ').split()))
    if len(block_sizes) != block_count or any((x <= 0 for x in block_sizes)):
        print('Enter valid positive sizes for all blocks.')
        raise SystemExit
    process_count = int(input('Enter number of processes: '))
    if process_count < 1 or process_count > ITEM_LIMIT:
        print('Number of processes must be between 1 and 20.')
        raise SystemExit
    process_sizes = list(map(int, input('Enter sizes of processes: ').split()))
    if len(process_sizes) != process_count or any((x <= 0 for x in process_sizes)):
        print('Enter valid positive sizes for all processes.')
        raise SystemExit
    first_fit(block_sizes, process_sizes)
    best_fit(block_sizes, process_sizes)
    worst_fit(block_sizes, process_sizes)

## Program 9: Page Replacement

Compare FIFO, LRU and Optimal page replacement using a common reference string.

**Sample input for the saved run:**

```text
12
7 0 1 2 0 3 0 4 2 3 0 3
3
```


In [ ]:
FRAME_LIMIT = 10
PAGE_LIMIT = 50

def page_is_loaded(page_frames, frame_count, page):
    for i in range(frame_count):
        if page_frames[i] == page:
            return True
    return False

def fifo_replacement(reference_string, frame_count):
    page_frames = [-1] * frame_count
    fifo_pointer = 0
    page_faults = 0
    for page in reference_string:
        if not page_is_loaded(page_frames, frame_count, page):
            page_frames[fifo_pointer] = page
            fifo_pointer = (fifo_pointer + 1) % frame_count
            page_faults += 1
    print('FIFO Page Faults  =', page_faults)

def lru_replacement(reference_string, frame_count):
    page_frames = [-1] * frame_count
    last_used = [0] * frame_count
    filled_count = 0
    page_faults = 0
    for i, page in enumerate(reference_string):
        frame_index = -1
        for j in range(frame_count):
            if page_frames[j] == page:
                frame_index = j
                break
        if frame_index == -1:
            page_faults += 1
            if filled_count < frame_count:
                page_frames[filled_count] = page
                last_used[filled_count] = i
                filled_count += 1
            else:
                lru_index = 0
                for j in range(1, frame_count):
                    if last_used[j] < last_used[lru_index]:
                        lru_index = j
                page_frames[lru_index] = page
                last_used[lru_index] = i
        else:
            last_used[frame_index] = i
    print('LRU Page Faults   =', page_faults)

def optimal_replacement(reference_string, frame_count):
    page_frames = [-1] * frame_count
    filled_count = 0
    page_faults = 0
    for i, page in enumerate(reference_string):
        if page_is_loaded(page_frames, filled_count, page):
            continue
        page_faults += 1
        if filled_count < frame_count:
            page_frames[filled_count] = page
            filled_count += 1
            continue
        replace_index = -1
        farthest_use = -1
        for j in range(frame_count):
            try:
                k = reference_string.index(page_frames[j], i + 1)
            except ValueError:
                replace_index = j
                break
            if k > farthest_use:
                farthest_use = k
                replace_index = j
        page_frames[replace_index] = page
    print('Optimal Page Faults =', page_faults)
if __name__ == '__main__':
    n = int(input('Enter number of pages in reference string: '))
    if n < 1 or n > PAGE_LIMIT:
        print('Number of pages must be between 1 and 50.')
        raise SystemExit
    reference_string = list(map(int, input('Enter the reference string: ').split()))
    if len(reference_string) != n or any((page < 0 for page in reference_string)):
        print('Enter exactly', n, 'valid non-negative page numbers.')
        raise SystemExit
    frame_count = int(input('Enter number of frames: '))
    if frame_count < 1 or frame_count > FRAME_LIMIT:
        print('Number of frames must be between 1 and 10.')
        raise SystemExit
    fifo_replacement(reference_string, frame_count)
    lru_replacement(reference_string, frame_count)
    optimal_replacement(reference_string, frame_count)

## Program 10: Disk Scheduling

Compare FCFS, SSTF, SCAN and C-SCAN by total head movement.

**Sample input for the saved run:**

```text
8
90 180 37 120 14 114 65 67
55
200
```


In [ ]:
REQUEST_LIMIT = 50

def head_distance(a, b):
    return abs(a - b)

def fcfs_scheduling(requests, request_count, initial_head):
    total_movement = 0
    current_head = initial_head
    for i in range(request_count):
        total_movement += head_distance(current_head, requests[i])
        current_head = requests[i]
    print('FCFS   total head movement =', total_movement)

def sstf_scheduling(requests, request_count, initial_head):
    visited_flags = [0] * request_count
    total_movement = 0
    current_head = initial_head
    for c in range(request_count):
        selected_index = -1
        shortest_distance = float('inf')
        for i in range(request_count):
            if not visited_flags[i] and head_distance(current_head, requests[i]) < shortest_distance:
                shortest_distance = head_distance(current_head, requests[i])
                selected_index = i
        visited_flags[selected_index] = 1
        total_movement += shortest_distance
        current_head = requests[selected_index]
    print('SSTF   total head movement =', total_movement)

def scan_scheduling(requests, request_count, initial_head, disk_size):
    ordered_requests = requests.copy()
    ordered_requests.append(initial_head)
    ordered_requests.sort()
    selected_index = ordered_requests.index(initial_head)
    total_movement = 0
    current_head = initial_head
    for i in range(selected_index + 1, len(ordered_requests)):
        total_movement += head_distance(current_head, ordered_requests[i])
        current_head = ordered_requests[i]
    total_movement += head_distance(current_head, disk_size - 1)
    current_head = disk_size - 1
    for i in range(selected_index - 1, -1, -1):
        total_movement += head_distance(current_head, ordered_requests[i])
        current_head = ordered_requests[i]
    print('SCAN   total head movement =', total_movement)

def cscan_scheduling(requests, request_count, initial_head, disk_size):
    ordered_requests = requests.copy()
    ordered_requests.append(initial_head)
    ordered_requests.sort()
    selected_index = ordered_requests.index(initial_head)
    total_movement = 0
    current_head = initial_head
    for i in range(selected_index + 1, len(ordered_requests)):
        total_movement += head_distance(current_head, ordered_requests[i])
        current_head = ordered_requests[i]
    total_movement += head_distance(current_head, disk_size - 1)
    total_movement += disk_size - 1
    current_head = 0
    for i in range(selected_index):
        total_movement += head_distance(current_head, ordered_requests[i])
        current_head = ordered_requests[i]
    print('C-SCAN total head movement =', total_movement)
if __name__ == '__main__':
    request_count = int(input('Enter number of requests: '))
    if request_count < 1 or request_count > REQUEST_LIMIT:
        print('Number of requests must be between 1 and 50.')
        raise SystemExit
    requests = list(map(int, input('Enter the requests: ').split()))
    if len(requests) != request_count:
        print('Please enter exactly', request_count, 'requests.')
        raise SystemExit
    initial_head = int(input('Enter initial head position: '))
    disk_size = int(input('Enter disk size (max cylinder + 1): '))
    if disk_size < 1 or not 0 <= initial_head < disk_size:
        print('Invalid disk size or initial head position.')
        raise SystemExit
    if any((r < 0 or r >= disk_size for r in requests)):
        print('All requests must be between 0 and', disk_size - 1)
        raise SystemExit
    fcfs_scheduling(requests, request_count, initial_head)
    sstf_scheduling(requests, request_count, initial_head)
    scan_scheduling(requests, request_count, initial_head, disk_size)
    cscan_scheduling(requests, request_count, initial_head, disk_size)

## Program 11: File Allocation

Demonstrate contiguous, linked and indexed allocation on a simulated disk.

**Sample input for the saved run:**

```text
2
2
3
4
```


In [ ]:
DISK_BLOCK_COUNT = 30
FILE_LIMIT = 10
disk_map = [0] * DISK_BLOCK_COUNT

def reset_disk_map():
    global disk_map
    disk_map = [0] * DISK_BLOCK_COUNT

def contiguous_allocation():
    reset_disk_map()
    print('\n--- Contiguous Allocation ---')
    file_count = int(input('Enter number of files: '))
    for file_number in range(file_count):
        starting_block, block_length = map(int, input(f'File {file_number + 1} - starting block, length: ').split())
        is_available = True
        if starting_block < 0 or block_length <= 0 or starting_block + block_length > DISK_BLOCK_COUNT:
            is_available = False
        else:
            for i in range(starting_block, starting_block + block_length):
                if disk_map[i] == 1:
                    is_available = False
                    break
        if not is_available:
            print('Allocation failed: blocks not free/out of range.')
            continue
        for i in range(starting_block, starting_block + block_length):
            disk_map[i] = 1
        print(f'File {file_number + 1} allocated blocks {starting_block} to {starting_block + block_length - 1}')

def linked_allocation():
    reset_disk_map()
    print('\n--- Linked Allocation ---')
    file_count = int(input('Enter number of files: '))
    for file_number in range(file_count):
        block_length = int(input(f'File {file_number + 1} - number of blocks: '))
        if block_length <= 0:
            print('Invalid number of blocks.')
            continue
        allocated_blocks = []
        for i in range(DISK_BLOCK_COUNT):
            if disk_map[i] == 0 and len(allocated_blocks) < block_length:
                disk_map[i] = 1
                allocated_blocks.append(i)
        print(f'File {file_number + 1} allocated blocks: ', end='')
        if allocated_blocks:
            print(' -> '.join(map(str, allocated_blocks)), end=' -> ')
        print('(NULL)')
        if len(allocated_blocks) < block_length:
            print('Warning: disk full, could not allocate all blocks.')

def indexed_allocation():
    reset_disk_map()
    print('\n--- Indexed Allocation ---')
    file_count = int(input('Enter number of files: '))
    for file_number in range(file_count):
        block_length = int(input(f'File {file_number + 1} - number of data blocks needed: '))
        if block_length < 0:
            print('Invalid number of blocks.')
            continue
        index_block = -1
        for i in range(DISK_BLOCK_COUNT):
            if disk_map[i] == 0:
                index_block = i
                disk_map[i] = 1
                break
        if index_block == -1:
            print(f'File {file_number + 1}: Allocation failed. No index block available.')
            continue
        data_blocks = []
        for i in range(DISK_BLOCK_COUNT):
            if disk_map[i] == 0 and len(data_blocks) < block_length:
                disk_map[i] = 1
                data_blocks.append(i)
        print(f'File {file_number + 1} - Index block: {index_block}, Data blocks: ', end='')
        print(*data_blocks)
        if len(data_blocks) < block_length:
            print('Warning: disk full, could not allocate all data blocks.')
if __name__ == '__main__':
    print('Menu:')
    print('1. Contiguous Allocation')
    print('2. Linked Allocation')
    print('3. Indexed Allocation')
    choice = int(input('Enter choice: '))
    if choice == 1:
        contiguous_allocation()
    elif choice == 2:
        linked_allocation()
    elif choice == 3:
        indexed_allocation()
    else:
        print('Invalid choice')